In [2]:
# =============================================================================
# feature_engineering.ipynb  —  Meta-Feature Extraction (Production-Grade)
# =============================================================================
# Reads:  cleaned_train.csv, cleaned_test.csv  (from process_data.ipynb)
# Writes: X_features_raw_train.npy, X_features_raw_test.npy
#         y_train.npy, y_test.npy
#
# Feature column → source text column:
#   char_count, word_count, capital_ratio, punctuation_ratio  → clean_raw_text
#   sentiment (TextBlob polarity)                             → bert_text
#
# MinMaxScaler is intentionally OMITTED here.
# Scaling will happen inside the sklearn Pipeline in train_model.ipynb,
# which is the only safe way to prevent test-set leakage from the scaler.
# =============================================================================

import pandas as pd
import numpy as np
from textblob import TextBlob
from tqdm.notebook import tqdm

In [3]:
# ── Load train / test splits ───────────────────────────────────────────────────
df_train = pd.read_csv("cleaned_train.csv")
df_test  = pd.read_csv("cleaned_test.csv")

print(f"Train: {len(df_train):,} rows  |  Test: {len(df_test):,} rows")
print(f"Columns: {df_train.columns.tolist()}")

In [4]:
# ── Feature extraction function ────────────────────────────────────────────────
# Feature layout (order matters — must match train_model.ipynb):
#   [0] char_count        — len(clean_raw_text)
#   [1] word_count        — token count
#   [2] capital_ratio     — uppercase chars / total chars
#   [3] punctuation_ratio — .!?,;: chars / total chars
#   [4] sentiment         — TextBlob polarity on bert_text  [-1, 1]
#
# Why separate source columns?
#   clean_raw_text retains original capitalisation and punctuation → reliable ratios.
#   bert_text retains grammar and natural language → reliable sentiment.

FEATURE_NAMES = ['char_count', 'word_count', 'capital_ratio', 'punctuation_ratio', 'sentiment']


def extract_features(df: pd.DataFrame) -> np.ndarray:
    records = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc="Meta-features"):
        raw  = str(row['clean_raw_text']) if pd.notna(row.get('clean_raw_text')) else ''
        bert = str(row['bert_text'])      if pd.notna(row.get('bert_text'))      else ''

        n = len(raw)
        char_count        = n
        word_count        = len(raw.split())
        capital_ratio     = sum(1 for c in raw if c.isupper()) / n if n else 0.0
        punctuation_ratio = sum(1 for c in raw if c in '.!?,;:') / n if n else 0.0
        try:
            sentiment = TextBlob(bert).sentiment.polarity
        except Exception:
            sentiment = 0.0

        records.append([char_count, word_count, capital_ratio, punctuation_ratio, sentiment])

    return np.array(records, dtype=np.float32)

🧠 Đang tính sentiment polarity...


🧠 Sentiment:   0%|          | 0/89712 [00:00<?, ?text/s]

➡️ Đã tính 50/89712 sentiment (0.06%)
➡️ Đã tính 100/89712 sentiment (0.11%)
➡️ Đã tính 150/89712 sentiment (0.17%)
➡️ Đã tính 200/89712 sentiment (0.22%)
➡️ Đã tính 250/89712 sentiment (0.28%)
➡️ Đã tính 300/89712 sentiment (0.33%)
➡️ Đã tính 350/89712 sentiment (0.39%)
➡️ Đã tính 400/89712 sentiment (0.45%)
➡️ Đã tính 450/89712 sentiment (0.50%)
➡️ Đã tính 500/89712 sentiment (0.56%)
➡️ Đã tính 550/89712 sentiment (0.61%)
➡️ Đã tính 600/89712 sentiment (0.67%)
➡️ Đã tính 650/89712 sentiment (0.72%)
➡️ Đã tính 700/89712 sentiment (0.78%)
➡️ Đã tính 750/89712 sentiment (0.84%)
➡️ Đã tính 800/89712 sentiment (0.89%)
➡️ Đã tính 850/89712 sentiment (0.95%)
➡️ Đã tính 900/89712 sentiment (1.00%)
➡️ Đã tính 950/89712 sentiment (1.06%)
➡️ Đã tính 1000/89712 sentiment (1.11%)
➡️ Đã tính 1050/89712 sentiment (1.17%)
➡️ Đã tính 1100/89712 sentiment (1.23%)
➡️ Đã tính 1150/89712 sentiment (1.28%)
➡️ Đã tính 1200/89712 sentiment (1.34%)
➡️ Đã tính 1250/89712 sentiment (1.39%)
➡️ Đã tính 1300/8971

In [5]:
# ── Compute features for train and test ────────────────────────────────────────
print("Computing train meta-features …")
X_features_train = extract_features(df_train)

print("\nComputing test meta-features …")
X_features_test = extract_features(df_test)

print(f"\nTrain features shape : {X_features_train.shape}")
print(f"Test  features shape : {X_features_test.shape}")
print(f"Feature names        : {FEATURE_NAMES}")
print(f"\nTrain sample (row 0) : {dict(zip(FEATURE_NAMES, X_features_train[0]))}")

🔢 Đang tạo các đặc trưng thống kê từ văn bản...


🔤 Stats:   0%|          | 0/89712 [00:00<?, ?text/s]

➡️ Đã xử lý 50/89712 dòng (0.06%)
➡️ Đã xử lý 100/89712 dòng (0.11%)
➡️ Đã xử lý 150/89712 dòng (0.17%)
➡️ Đã xử lý 200/89712 dòng (0.22%)
➡️ Đã xử lý 250/89712 dòng (0.28%)
➡️ Đã xử lý 300/89712 dòng (0.33%)
➡️ Đã xử lý 350/89712 dòng (0.39%)
➡️ Đã xử lý 400/89712 dòng (0.45%)
➡️ Đã xử lý 450/89712 dòng (0.50%)
➡️ Đã xử lý 500/89712 dòng (0.56%)
➡️ Đã xử lý 550/89712 dòng (0.61%)
➡️ Đã xử lý 600/89712 dòng (0.67%)
➡️ Đã xử lý 650/89712 dòng (0.72%)
➡️ Đã xử lý 700/89712 dòng (0.78%)
➡️ Đã xử lý 750/89712 dòng (0.84%)
➡️ Đã xử lý 800/89712 dòng (0.89%)
➡️ Đã xử lý 850/89712 dòng (0.95%)
➡️ Đã xử lý 900/89712 dòng (1.00%)
➡️ Đã xử lý 950/89712 dòng (1.06%)
➡️ Đã xử lý 1000/89712 dòng (1.11%)
➡️ Đã xử lý 1050/89712 dòng (1.17%)
➡️ Đã xử lý 1100/89712 dòng (1.23%)
➡️ Đã xử lý 1150/89712 dòng (1.28%)
➡️ Đã xử lý 1200/89712 dòng (1.34%)
➡️ Đã xử lý 1250/89712 dòng (1.39%)
➡️ Đã xử lý 1300/89712 dòng (1.45%)
➡️ Đã xử lý 1350/89712 dòng (1.50%)
➡️ Đã xử lý 1400/89712 dòng (1.56%)
➡️ Đã xử lý 

In [6]:
# ── Save raw feature arrays and labels ────────────────────────────────────────
# NO SCALING here — MinMaxScaler belongs inside the sklearn Pipeline
# (fitted on train, applied to both train and test consistently).

np.save("X_features_raw_train.npy", X_features_train)
np.save("X_features_raw_test.npy",  X_features_test)
np.save("y_train.npy", df_train['label'].values.astype(np.int32))
np.save("y_test.npy",  df_test['label'].values.astype(np.int32))

print("Saved:")
print("  X_features_raw_train.npy  shape:", X_features_train.shape)
print("  X_features_raw_test.npy   shape:", X_features_test.shape)
print("  y_train.npy               shape:", df_train['label'].shape)
print("  y_test.npy                shape:", df_test['label'].shape)
print("\nNOTE: MinMaxScaler will be applied inside the Pipeline in train_model.ipynb")

📏 Đang chuẩn hóa các đặc trưng phụ...


In [7]:
# ── Distribution sanity check ──────────────────────────────────────────────────
_df_check = pd.DataFrame(X_features_train, columns=FEATURE_NAMES)
print("Train feature statistics (raw, unscaled):")
print(_df_check.describe().round(4))

✅ Đã lưu đặc trưng nâng cao vào: X_features_scaled.npy và cleaned_data_with_features.csv
